# L6 — CloudWatch Omni: Agent Observability

## What you'll learn

- How to turn on Amazon CloudWatch Omni in your account and open its sign-in page
- How to see the orchestrator's traces and your CloudWatch metrics together in Omni
- How to ask the Omni AI assistant about your agents' behavior in plain language

## Overview

CloudWatch Omni is an AI-powered way to explore your telemetry, built on top of CloudWatch. It opens on its own web page, outside the AWS Console, and reads the same CloudWatch data you already have. There is nothing to re-instrument: the traces the orchestrator already sends appear in Omni as soon as a Space exists.

Setup has two parts:

- **Domain.** Your organization's entry point, with a fixed sign-in URL. It can cover one account or a whole AWS Organization.
- **Space.** Where you investigate. A Space covers one account and Region, holds the telemetry you can see, and controls who has access.

This notebook creates a Domain and a Space in your current account and Region.

## Architecture

![Observability layer: the agents emit OpenTelemetry spans. By default they go through managed ADOT to CloudWatch and CloudWatch Omni; with the Datadog option an OTLP exporter in the agent sends them to Datadog Agent Observability.](images/l6-observability-architecture.png)

This notebook follows the CloudWatch path at the top of the diagram: the agents' spans go to CloudWatch, and Omni reads them there.

## Steps

0. Install dependencies
1. Shared setup
2. How CloudWatch Omni fits with CloudWatch
3. Set up CloudWatch Omni
4. Generate traces to view in Omni
5. Open CloudWatch Omni and investigate

## License details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- L3 ([`3_orchestrator_agent.ipynb`](../l3-orchestration/3_orchestrator_agent.ipynb)) is complete, with CloudWatch as the observability provider (the default).
- You can create IAM roles and CloudWatch Omni resources in the account.
- Your console role can become a Space admin. Step 3 does this for you when the workshop stack has your role; if it doesn't, add the role yourself in Omni → Space admins.

## Step 0: Install Dependencies


`cloudwatchomni` is available in boto3 1.43.101 and later. The install is idempotent - safe to re-run.


In [ ]:
%pip install --quiet --upgrade "boto3>=1.43.101"


## Step 1: Shared setup

Resolve region, caller identity, and the Harness ARN from SSM.


In [ ]:
import os, boto3
from helper.cloudwatch_omni_setup import setup

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
ACCOUNT_ID = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]

ssm = boto3.client("ssm", region_name=REGION)
try:
    HARNESS_ARN = ssm.get_parameter(Name="/anycompany/agentcore/harness_arn")["Parameter"]["Value"]
except ssm.exceptions.ParameterNotFound as _e:
    raise RuntimeError(
        "Harness ARN not found at /anycompany/agentcore/harness_arn. "
        "Run the l3-orchestration notebooks (1_gateway_tools through "
        "3_orchestrator_agent) first — Step 7 of 3_orchestrator_agent "
        "publishes this key to SSM."
    ) from _e

# Harness client is only needed by Step 4 (generate a trace); define it here so
# the rest of the notebook can refer to it.
runtime = boto3.client("bedrock-agentcore", region_name=REGION)

print(f"Region:       {REGION}")
print(f"Account:      {ACCOUNT_ID}")
print(f"Harness ARN:  {HARNESS_ARN}")


## Step 2: How CloudWatch Omni fits with CloudWatch

Omni is a *new experience* on top of the existing CloudWatch telemetry store. It does not replace anything.

| | CloudWatch (console) | CloudWatch Omni |
|---|---|---|
| **Where you work** | AWS Console (per account, per Region) | Dedicated URL outside the Console; one URL for your Domain |
| **What you query** | Logs Insights, Metrics, X-Ray, Service Lens | Same data, but with an AI assistant that writes the query in plain language |
| **Agent observability** | Spans + metrics via AgentCore | Spans + metrics + continuous AgentCore Evaluations scored alongside |
| **Instrumentation** | OpenTelemetry / AWS SDKs | Same (no re-instrumentation) |

The Harness emits OpenTelemetry spans to CloudWatch as soon as it processes a request. Step 4 below invokes the Harness so there is fresh telemetry to view. Enabling Omni gives you a different UI on top of the same data.


## Step 3: Set up CloudWatch Omni

One call provisions everything the Space needs to ingest and surface agent telemetry:

1. **Space operator role** - IAM role trusted by `cloudwatch.amazonaws.com`, scoped to this account, with the three AWS-managed Omni policies attached.
2. **Account Domain + Space** - the sign-in URL and the Space your telemetry flows into (one Space per account+region).
3. **SPACE_ADMIN grant** - adds the IAM role from the `ParticipantRoleArn` stack parameter (your console sign-in identity) so you can open the sign-in URL without the "not a member" banner.
4. **Telemetry integrations** - Dataset Integration role (trusted by `logs.amazonaws.com`) and the two enrichment flips the Space needs to read your telemetry.

Every underlying call is idempotent, so this cell is safe to re-run. See [`helper/cloudwatch_omni_setup.py`](helper/cloudwatch_omni_setup.py) for the full mechanics.


In [ ]:
# One-call setup: role -> domain -> space -> admin grant -> telemetry integrations.
result = setup(region=REGION, account_id=ACCOUNT_ID)

DOMAIN_ID    = result["domain_id"]
DOMAIN_URL   = result["domain_url"]
SPACE_ID     = result["space_id"]
SPACE_ARN    = result["space_arn"]
DI_ROLE_ARN  = result["di_role_arn"]

print()
print("=" * 60)
print("CloudWatch Omni is set up.")
print("=" * 60)
print(f"Sign-in URL: {DOMAIN_URL}")
print(f"Space:       {SPACE_ID} ({REGION})")
print()
print("Wait ~5 min for ingestion to backfill, then open the sign-in URL and")
print("check the Services canvas - all agents should show as 'Instrumented'.")


## Step 4: Generate traces to view in Omni

Invoke the Harness three times in a single session with questions that force it to delegate to each sub-agent at least once: an order lookup (routes to `order_agent_a2a`) and two refund queries (route to `refund_agent_a2a`). After this cell runs, Omni's Agents view shows both sub-agents alongside the Harness. The Harness already emits OpenTelemetry spans to CloudWatch; they appear in your Space automatically.

Note the printed `session_id` - you'll search for it inside Omni in the next step.


In [ ]:
import time, uuid

# AgentCore invoke_harness requires runtimeSessionId to be 33-100 characters.
# Pad a human-readable "omni-demo-<timestamp>-" prefix with a uuid4 hex so the
# session is easy to spot in the Omni UI but still passes validation. Reuse the
# same session_id across all three turns so Omni shows one multi-turn session.
session_id = f"omni-demo-{int(time.time())}-{uuid.uuid4().hex}"
actor_id   = f"workshop-user-{ACCOUNT_ID[-4:]}"
print(f"Session ID: {session_id}")
print(f"Actor ID:   {actor_id}")

# Three turns that force the Harness to delegate to BOTH sub-agents at least
# once, so Omni's Agents view shows order_agent_a2a AND refund_agent_a2a:
#   - Turn 1 -> order lookup  -> routes to order_agent_a2a
#   - Turn 2 -> refund policy -> routes to refund_agent_a2a
#   - Turn 3 -> refund action -> routes to refund_agent_a2a (typically check_eligible + process_refund)
QUESTIONS = [
    "Look up order ORD-10001 for customer CUST-789.",
    "What is your refund policy for electronics?",
    "I want to refund ORD-10001 because it was damaged on arrival.",
]

for turn, question in enumerate(QUESTIONS, start=1):
    print()
    print("=" * 72)
    print(f"Turn {turn}: {question}")
    print("=" * 72)
    response = runtime.invoke_harness(
        harnessArn=HARNESS_ARN,
        runtimeSessionId=session_id,
        actorId=actor_id,
        messages=[{"role": "user", "content": [{"text": question}]}],
    )
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            if "text" in delta:
                print(delta["text"], end="", flush=True)
    print()

print()
print("-" * 72)
print(f"Done. In Omni, filter by session_id = {session_id}")
print("to see this three-turn conversation and the sub-agent calls underneath.")


## Step 5: Open CloudWatch Omni and investigate

1. **Open the sign-in URL** printed in Step 3 (`https://anycompany-<last 4 digits of your account ID>.cloudwatch-omni.global.app.aws`). When prompted *"How do you sign in to AWS?"*, pick **"Root user or IAM user"** and click **Continue to AWS sign-in** - the SPACE_ADMIN grant was issued to the IAM role from the `ParticipantRoleArn` stack parameter, so this is the identity Omni expects.

Once you're signed in, walk through the Omni UI in this order.

### 5.1 Home page

The Omni home page is a canvas picker. Each card is a different lens on your telemetry (Services, Agents, Logs, Metrics, Playground). Everything in this walkthrough starts from here.

![CloudWatch Omni home page with setup cards and tool links grouped under Application performance monitoring, Analytics and dashboards, and Agent observability](images/cloudwatch_omni_homepage.png)

### 5.2 Services view

Lists every service Omni discovered in this account. Confirm the Harness and both sub-agents are present and marked **Instrumented**: `anycompany_orchestrator`, `order_agent_a2a`, `refund_agent_a2a`. If any show *Not Instrumented*, the Dataset Integration from Step 3 is still backfilling - give it ~5 minutes and refresh.

![Services view with Instrumented agents](images/cloudwatch_omni_servicesview.png)

### 5.3 Agent view

Click an agent (for example `anycompany_orchestrator`) to open its agent view. You get RED metrics (rate, errors, duration), the model it invokes, the tools it calls, and a list of recent sessions.

![Agent view with RED metrics](images/cloudwatch_omni_agentview.png)

### 5.4 Agent topology

Switch to the topology tab to see how the Harness delegates to the sub-agents and which tools each one calls. Useful for spotting an agent that was wired up but is never actually invoked.

![Agent topology flow for the orchestrator Harness showing the agent step, Claude Sonnet 4.6 LLM calls, and agent-invoker, registry-discovery and code_interpreter tool calls with trace counts](images/cloudwatch_omni_agenttopology.png)

### 5.5 Agent traces

Filter the traces list by the `session_id` printed in Step 4 to find the three-turn demo you just ran. Open any trace to see the full span waterfall - the Harness's reasoning, each tool call, each sub-agent invocation, and token counts per model call.

![Agent traces list with one trace open in Graph view, showing AgentCore retrieval and chat spans, and the span overview with model, token counts and GenAI attributes](images/cloudwatch_omni_agenttraces.png)

### Try the AI assistant

Use the chat affordance at the bottom-right of any trace or service view to ask questions in plain language. Grab a `trace_id` from one of the traces you opened in Step 5.5 and try:

> *Provide token analysis for trace `<trace_id>`.*

The assistant writes the SQL/PromQL query against your telemetry, runs it, and shows the result inline. It also shows the query it ran so the answer is auditable.

![Omni AI assistant - token analysis for a trace](images/cloudwatch_omni_chat.png)

Other prompts worth trying:
- *"Summarize what this agent did on this request."*
- *"Did any tool call take longer than expected?"*
- *"Show me all failed requests from this agent in the last hour."*

### (Optional) Set up a continuous evaluator

Under *Agent evaluations*, enable AgentCore Evaluations against this Space to score every production response against criteria like helpfulness, scope adherence, or custom rules you define. Scored results appear alongside latency and token counts on each trace.


## Summary

This notebook turned on CloudWatch Omni in your account, with a Domain and a Space named `anycompany-` plus the last four digits of your account ID. It reused the traces the agents already send, and you followed an agent run through Omni, including its AI assistant for questions in plain language.

**Next:** [`1_agent_evaluations.ipynb`](../l7-agentic-operations/1_agent_evaluations.ipynb). Score how well the agents handle conversations.

---
## Cleanup

Run the following cell (after uncommenting) to tear down everything `setup()` created: the Space, access grants, Domain, Dataset Integration, and both IAM roles (`CloudWatchOmniOperatorRole`, `CloudWatchOmniDatasetIntegrationRole`). See [`helper/cloudwatch_omni_setup.py`](helper/cloudwatch_omni_setup.py) `teardown()` for the full order.


In [ ]:
## === CLEANUP: Delete the CloudWatch Omni resources created by this notebook ===
## Uncomment and run only after you are done with the Omni UI.
##
## Deletes (in reverse dependency order):
##   - access grants on the Space
##   - the Space
##   - the Domain (set delete_domain=False to keep it)
##   - the Dataset Integration
##   - CloudWatchOmniDatasetIntegrationRole + inline policy
##   - CloudWatchOmniOperatorRole + attached managed policies
##
## The telemetry + OTel enrichments are account-level and often shared with
## other work, so they are left running by default. Pass stop_enrichments=True
## to turn them off too.

# from helper.cloudwatch_omni_setup import teardown
# teardown(region=REGION, account_id=ACCOUNT_ID)
